# Ustoz chat: Kaggle backend (GPU)

Bu notebook chat sahifasi uchun **backend**: Qwen2.5-1.5B + **DPO v2** (yakuniy model) + **RLHF (Gemini hakam)**,
Kaggle GPU'da. Javob ~2-3 soniya (lokal CPU'da ~20 s). Sahifa (frontend) Vercel'da:
https://teach-help.vercel.app

```
Vercel sahifa  --?api=-->  Cloudflare tunnel  -->  shu notebook (app.py, GPU)
```

**Boshlashdan oldin:**
- **Settings -> Accelerator -> GPU T4 x2** (bitta GPU ham yetadi), **Internet -> On**
- **Add Input** -> **`ustoz-chat-adapters`** (sizning shaxsiy datasetingiz: DPO v2 va RLHF adapterlari)

**Ishga tushirish: editor'da Run All** (Save Version emas). Oxirgi katakda havola chiqadi:
`https://teach-help.vercel.app/?api=https://....trycloudflare.com` — shuni oching yoki talabalarga yuboring.
Brauzer oynasi ochiq tursin; tugatganda **⏻ (Stop session)** bosing: GPU soatlari tejaladi.

Eslatma: Kaggle — data science uchun platforma, doimiy hosting emas. Dars yoki namoyish uchun ishlating.

## 1. GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 2. Kod, kutubxonalar va base model

In [ ]:
%cd /kaggle/working
!rm -rf Post_Training
!GIT_LFS_SKIP_SMUDGE=1 git clone -q https://github.com/davronbek-malikov/Post_Training.git
%cd Post_Training
!pip -q install -U transformers peft fastapi uvicorn
!pip -q uninstall -y torchao  # Kaggle'dagi eski torchao peft bilan adapter yuklashni buzadi

from huggingface_hub import snapshot_download
snapshot_download("Qwen/Qwen2.5-1.5B", local_dir="Qwen2.5-1.5B")

## 3. Adapterlar (`ustoz-chat-adapters` datasetidan)

In [ ]:
import glob, os, shutil, zipfile
ADAPTERS = "/kaggle/working/adapters"
os.makedirs(ADAPTERS, exist_ok=True)
for model_id in ("sft-lora-dpo-v2", "sft-lora-rlhf-llm"):
    found = glob.glob(f"/kaggle/input/**/{model_id}/adapter_config.json", recursive=True)
    if found:
        shutil.copytree(os.path.dirname(found[0]), f"{ADAPTERS}/{model_id}", dirs_exist_ok=True)
    else:   # dataset zip ko'rinishida qo'shilgan bo'lsa
        for z in glob.glob(f"/kaggle/input/**/{model_id}.zip", recursive=True):
            zipfile.ZipFile(z).extractall(f"{ADAPTERS}/{model_id}")
    print(model_id, "->", os.listdir(f"{ADAPTERS}/{model_id}") if os.path.isdir(f"{ADAPTERS}/{model_id}") else "TOPILMADI")
assert all(os.path.exists(f"{ADAPTERS}/{m}/adapter_config.json") for m in ("sft-lora-dpo-v2", "sft-lora-rlhf-llm")), \
    "Adapterlar topilmadi: Add Input -> ustoz-chat-adapters ni qo'shing"

## 4. Backend (app.py) — GPU'da

In [ ]:
import subprocess, time, json, urllib.request
env = {**os.environ, "BASE_PATH": "/kaggle/working/Post_Training/Qwen2.5-1.5B", "ADAPTERS_PATH": ADAPTERS}
backend_log = open("/kaggle/working/backend.log", "w")
backend = subprocess.Popen(["python", "-m", "uvicorn", "app:app", "--host", "127.0.0.1", "--port", "7860"],
                           cwd="deploy/hf_space", env=env, stdout=backend_log, stderr=subprocess.STDOUT)
for _ in range(120):   # model yuklanguncha (GPU'da ~1 daqiqa)
    if backend.poll() is not None:
        raise SystemExit("Backend to'xtadi:\n" + open("/kaggle/working/backend.log").read()[-3000:])
    try:
        models = json.load(urllib.request.urlopen("http://127.0.0.1:7860/api/models", timeout=5))
        print("Backend tayyor:", [m["id"] for m in models])
        break
    except OSError:
        time.sleep(5)
else:
    raise SystemExit("Backend 10 daqiqada ishga tushmadi")

## 5. Tunnel va havola

In [ ]:
import re
!wget -q -O /kaggle/working/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x /kaggle/working/cloudflared
tunnel = subprocess.Popen(["/kaggle/working/cloudflared", "tunnel", "--no-autoupdate", "--url", "http://127.0.0.1:7860"],
                          stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
url = None
for line in tunnel.stdout:
    m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", line)
    if m:
        url = m.group(0)
        break
link = f"https://teach-help.vercel.app/?api={url}"
open("/kaggle/working/backend_link.txt", "w").write(link + "\n")
print("=" * 70)
print("TAYYOR. Shu havolani oching yoki talabalarga yuboring:\n")
print("  ", link)
print("=" * 70)

## 6. Ishlab turish

Bu katak backendni `HOURS` soat ushlab turadi va har 10 daqiqada holatini yozadi. Tugatish: **Stop** (■) yoki
**⏻ Stop session**. Kaggle bitta sessiyani ko'pi bilan 12 soat ishlatadi.

In [ ]:
HOURS = 3
end = time.time() + HOURS * 3600
while time.time() < end:
    alive = backend.poll() is None and tunnel.poll() is None
    print(time.strftime("%H:%M"), "| backend", "ishlayapti" if alive else "TO'XTAGAN", "|", link, flush=True)
    if not alive:
        print(open("/kaggle/working/backend.log").read()[-2000:])
        break
    time.sleep(600)
tunnel.terminate(); backend.terminate()